# FloodRisk · Primer modelo experimental

**Ejecutar en Google Colab, con CPU.** Menú Archivo → Subir notebook. Ejecuta las celdas en orden.
En la carga selecciona el ZIP original de datasets, no el ZIP de auditoría. No necesitas LM Studio,
GPU, claves de API ni montar Drive. El ZIP se lee en memoria y no se modifica.

## Qué aprenderá
Clasificar si un condado-año registra **al menos un evento relacionado con inundaciones** según
`flood_related_event_count > 0`. El diccionario incluye inundación, huracán, marejada y lluvia fuerte.
**No es una etiqueta exclusiva de inundación, una predicción diaria ni una estimación de daños.**

Para la primera versión se utilizan condados-año presentes en la tabla NOAA. Un cero significa
que esa fila tiene cero eventos relacionados, no ausencia comprobada de inundaciones. Las filas
NOAA ausentes quedan desconocidas y no se crean negativos artificiales. Esto selecciona una
población con registros de tormentas: las métricas y probabilidades son de ese universo observado,
no una estimación validada para todos los condados. Se necesita un catálogo/cobertura completos
para ampliar el alcance. No se han validado todos los códigos contra un catálogo oficial.

El notebook compara una referencia constante, regresión logística y árboles potenciados.
Muestra métricas, curvas, predicciones y exporta un modelo reutilizable.

In [ ]:
# Dependencias de Colab. Sin actualizar paquetes si ya cumplen estos mínimos.
%pip -q install "pandas>=2.2,<4" "numpy>=1.26,<3" "scikit-learn>=1.5,<1.8" "matplotlib>=3.8,<4" "joblib>=1.3,<2"

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '2'
import io, json, zipfile, hashlib, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             precision_score, recall_score, f1_score, confusion_matrix)
from sklearn.calibration import CalibrationDisplay
SEED = 42
print('Versiones:', pd.__version__, np.__version__, sklearn.__version__)

## 1. Configuración temporal
Por defecto se predice **2027 con antecedentes hasta 2025**, sin utilizar el año parcial 2026.
Esto requiere aprender el mismo desfase de dos años en toda la historia: antecedentes hasta 2020
→ etiqueta 2022; hasta 2021 → etiqueta 2023, etc. No se predice 2026 para usar su resultado como entrada.

La validación selecciona el modelo con menor Brier (menor es mejor); las pruebas 2024–2025
se consultan después. Al final se reentrena el modelo seleccionado con etiquetas hasta 2025.
El resultado de prueba corresponde al entrenamiento anterior, no a una nueva prueba del modelo final.

**Límite de esta simulación:** los archivos son una instantánea revisada en agosto de 2026.
Usar años previos evita introducir eventos de años futuros, pero no reproduce exactamente cuándo
se publicaron o corrigieron históricamente los registros. No existen instantáneas de publicación
para garantizar un backtest «tal como se conocía entonces».

In [ ]:
TARGET_YEAR = 2027
LAST_COMPLETE_YEAR = 2025  # Último año calendario cerrado; puede tener revisiones/faltantes.
HORIZON = TARGET_YEAR - LAST_COMPLETE_YEAR
TRAIN_END = 2022
VALIDATION_YEAR = 2023
TEST_YEARS = [2024, 2025]
FIRST_TARGET_YEAR = 2005
CLASSIFICATION_THRESHOLD = 0.5  # Para métricas binarias; no umbral de protección civil.
assert HORIZON >= 1
assert TRAIN_END < VALIDATION_YEAR < min(TEST_YEARS)
assert max(TEST_YEARS) <= LAST_COMPLETE_YEAR
print(f'Objetivo {TARGET_YEAR}; antecedentes hasta {LAST_COMPLETE_YEAR}; desfase {HORIZON} años.')

In [ ]:
# En Colab abre un selector. Para ejecución local, asigna ZIP_PATH antes de ejecutar esta celda.
if not globals().get('ZIP_PATH'):
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Sube exactamente un ZIP con los datasets originales.')
    ZIP_PATH = next(iter(uploaded))
    del uploaded
ZIP_PATH = Path(ZIP_PATH)
zip_sha256 = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
with zipfile.ZipFile(ZIP_PATH) as archive:
    members = [n for n in archive.namelist() if not n.startswith('__MACOSX/') and not n.endswith('/')]
    def read_member(basename):
        matches = [n for n in members if Path(n).name == basename]
        if len(matches) != 1:
            raise ValueError(f'Se esperaba una copia de {basename}; encontradas: {len(matches)}')
        info = archive.getinfo(matches[0])
        if info.file_size > 100_000_000:
            raise ValueError('Archivo demasiado grande para esta versión del notebook.')
        return archive.read(matches[0])
    manifest = json.loads(read_member('source_manifest.json'))
    required = ['storm_events_county_year.csv', 'nfip_claims_county_year.csv',
                'fema_declarations_county_year.csv', 'data_dictionary.csv']
    raw_tables = {}
    for name in required:
        payload = read_member(name)
        expected = manifest.get('outputs', {}).get(name, {}).get('sha256')
        if expected and hashlib.sha256(payload).hexdigest() != expected:
            raise ValueError(f'Hash distinto al manifiesto: {name}')
        raw_tables[name] = pd.read_csv(io.BytesIO(payload), dtype={'county_fips':'string'})
print('Instantánea declarada:', manifest.get('generated_at_utc'))
display(pd.DataFrame([{'archivo':n, 'filas':len(d), 'columnas':len(d.columns)}
                      for n,d in raw_tables.items()]))
display(raw_tables['data_dictionary.csv'].query("column == 'flood_related_event_count'"))

## 2. Preparación sin modificar los originales
Se conservan identificadores de cinco dígitos con ceros iniciales, se excluyen códigos `XX000`,
y se limita el alcance inicial a los 50 estados y Washington D. C. Los territorios quedan fuera.
Este filtro no sustituye validar un catálogo de condados/equivalentes y sus cambios históricos.
Las uniones se validan como uno a uno. Las ausencias de cada fuente permanecen como faltantes.

La pérdida NFIP negativa se conserva en la copia original y se marca. Para las entradas del modelo
se trata como desconocida, pendiente de interpretar el ajuste contable. Se excluyen índices globales,
perfil «latest», `flood_zone_count` constante y datos del año objetivo. No se usan nombres de estado
inconsistentes: el estado se obtiene del prefijo FIPS como categoría.

In [ ]:
STATE_CODES = set('01 02 04 05 06 08 09 10 11 12 13 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 44 45 46 47 48 49 50 51 53 54 55 56'.split())
quality_rows = []
def clean_table(name):
    d = raw_tables[name].copy()
    d['county_fips'] = d.county_fips.str.strip().str.zfill(5)
    valid = (d.county_fips.str.fullmatch(r'\d{5}', na=False)
             & ~d.county_fips.str.endswith('000', na=False)
             & d.county_fips.str[:2].isin(STATE_CODES))
    quality_rows.append({'tabla':name, 'filas_originales':len(d), 'excluidas_geografia':int((~valid).sum())})
    d = d.loc[valid].copy()
    d['year'] = pd.to_numeric(d.year, errors='raise').astype(int)
    if d.duplicated(['county_fips','year']).any():
        raise ValueError(f'Claves repetidas: {name}')
    return d
storms = clean_table('storm_events_county_year.csv')
claims = clean_table('nfip_claims_county_year.csv')
fema = clean_table('fema_declarations_county_year.csv')
for col in ['storm_event_count', 'flood_related_event_count']:
    if storms[col].isna().any() or (storms[col] < 0).any():
        raise ValueError(f'Conteo inválido: {col}')
assert (storms.flood_related_event_count <= storms.storm_event_count).all()
claims['nfip_negative_adjustment'] = (claims.nfip_paid_loss_usd < 0).astype(int)
claims.loc[claims.nfip_paid_loss_usd < 0, 'nfip_paid_loss_usd'] = np.nan
storm_cols = ['storm_event_count','flood_related_event_count','storm_total_damage_usd',
              'storm_injuries','storm_deaths']
claim_cols = ['nfip_claim_count','nfip_paid_loss_usd','high_severity_claim_count','nfip_negative_adjustment']
fema_cols = ['fema_declaration_count']
history = storms[['county_fips','year'] + storm_cols].merge(
    claims[['county_fips','year'] + claim_cols], on=['county_fips','year'], how='outer', validate='one_to_one'
).merge(fema[['county_fips','year'] + fema_cols], on=['county_fips','year'], how='outer', validate='one_to_one')
history = history.loc[history.year <= LAST_COMPLETE_YEAR].copy()
display(pd.DataFrame(quality_rows))
coverage = storms.groupby('year').agg(filas=('county_fips','size'),
    positivos=('flood_related_event_count',lambda s: int((s>0).sum())))
coverage['proporcion_positiva'] = coverage.positivos / coverage.filas
display(coverage.tail(8))
coverage['filas'].plot(title='Cobertura NOAA observada por año (2026 parcial)', ylabel='Filas'); plt.show()

## 3. Ejemplos supervisados y antecedentes
Cada etiqueta corresponde al año objetivo. Las entradas corresponden exactamente al año base
y los dos anteriores; una fila faltante no se sustituye por «la última fila», que podría ser de otro año.
Se requiere un registro NOAA en el año base para mantener una población de emisión definida.
Las entradas monetarias se transforman con `log1p`. La imputación se ajusta dentro del entrenamiento.

In [ ]:
value_cols = storm_cols + claim_cols + fema_cols
def make_features(requests):
    out = requests[['county_fips','target_year']].copy()
    out['base_year'] = out.target_year - HORIZON
    out['state_fips'] = out.county_fips.str[:2].astype(str)
    for lag in range(3):
        lookup = history[['county_fips','year'] + value_cols].copy()
        lookup = lookup.rename(columns={'year':f'source_year_{lag}',
            **{c:f'{c}_lag{lag}' for c in value_cols}})
        out[f'lookup_year_{lag}'] = out.base_year - lag
        out = out.merge(lookup, how='left', left_on=['county_fips',f'lookup_year_{lag}'],
            right_on=['county_fips',f'source_year_{lag}'], validate='many_to_one')
        observed = out[f'source_year_{lag}'].notna()
        assert (out.loc[observed,f'source_year_{lag}'] <= out.loc[observed,'base_year']).all()
        assert (out.loc[observed,f'source_year_{lag}'] < out.loc[observed,'target_year']).all()
        out = out.drop(columns=[f'lookup_year_{lag}',f'source_year_{lag}'])
    out['noaa_history_years'] = sum(out[f'storm_event_count_lag{k}'].notna().astype(int) for k in range(3))
    for c in [c for c in out if '_usd_lag' in c]:
        out[c] = np.log1p(out[c].where(out[c] >= 0))
    return out

labels = storms.loc[storms.year.between(FIRST_TARGET_YEAR,LAST_COMPLETE_YEAR),
                    ['county_fips','year','flood_related_event_count']].copy()
labels = labels.rename(columns={'year':'target_year'})
labels['y'] = (labels.flood_related_event_count > 0).astype(int)
dataset = make_features(labels).merge(labels[['county_fips','target_year','y']],
    on=['county_fips','target_year'],validate='one_to_one')
before = len(dataset)
dataset = dataset.loc[dataset.storm_event_count_lag0.notna()].reset_index(drop=True)
feature_cols = [c for c in dataset if '_lag' in c] + ['noaa_history_years','state_fips']
assert 'y' not in feature_cols and 'target_year' not in feature_cols
assert dataset.base_year.max() <= LAST_COMPLETE_YEAR
print('Ejemplos utilizables:',len(dataset), '| Sin NOAA en año base:',before-len(dataset))
display(dataset[['county_fips','target_year','base_year','flood_related_event_count_lag0','y']].head())

In [ ]:
train = dataset.loc[dataset.target_year <= TRAIN_END].copy()
validation = dataset.loc[dataset.target_year == VALIDATION_YEAR].copy()
test = dataset.loc[dataset.target_year.isin(TEST_YEARS)].copy()
for name, part in [('entrenamiento',train),('validacion',validation),('prueba',test)]:
    assert len(part) > 0 and part.y.nunique() == 2, f'Partición insuficiente: {name}'
    print(name, len(part), 'filas; años', sorted(part.target_year.unique()), 'positivos', round(part.y.mean(),3))
assert train.target_year.max() < validation.target_year.min() < test.target_year.min()
numeric_cols = [c for c in feature_cols if c != 'state_fips']
def preprocessing():
    return ColumnTransformer([
        ('numeric',Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True)),
                             ('scale',StandardScaler())]),numeric_cols),
        ('state',OneHotEncoder(handle_unknown='ignore',sparse_output=False),['state_fips'])
    ], sparse_threshold=0)
templates = {
    'referencia_constante':DummyClassifier(strategy='prior'),
    'logistica':Pipeline([('prepare',preprocessing()),('model',LogisticRegression(max_iter=1500,random_state=SEED))]),
    'arboles_potenciados':Pipeline([('prepare',preprocessing()),('model',HistGradientBoostingClassifier(
        max_iter=100,max_leaf_nodes=15,min_samples_leaf=40,l2_regularization=2,
        early_stopping=False,random_state=SEED))])
}
def metrics(y, p):
    binary = np.asarray(p) >= CLASSIFICATION_THRESHOLD
    two_classes = len(np.unique(y)) == 2
    return {'roc_auc':roc_auc_score(y,p) if two_classes else np.nan,
            'average_precision':average_precision_score(y,p) if two_classes else np.nan,
            'brier':brier_score_loss(y,p),
            'precision':precision_score(y,binary,zero_division=0),
            'recall':recall_score(y,binary,zero_division=0),
            'f1':f1_score(y,binary,zero_division=0)}
fitted, validation_rows = {}, []
for name, template in templates.items():
    model = clone(template).fit(train[feature_cols],train.y)
    fitted[name] = model
    p = model.predict_proba(validation[feature_cols])[:,1]
    validation_rows.append({'modelo':name,**metrics(validation.y,p)})
validation_metrics = pd.DataFrame(validation_rows).sort_values('brier')
selected_name = validation_metrics.iloc[0].modelo
selected_model = fitted[selected_name]
display(validation_metrics)
print('Seleccionado SOLO con validación:', selected_name)

## 4. Prueba histórica que no selecciona el modelo
ROC-AUC y average precision resumen discriminación (más alto es mejor). Average precision no es
exactamente el área trapezoidal de la curva precision-recall. Brier mide error probabilístico
(más bajo es mejor). Se compara contra la referencia constante y la prevalencia observada.
La curva de calibración ayuda a ver si los porcentajes coinciden con las frecuencias observadas;
**mostrar una curva no significa que se haya calibrado el modelo**. No hay precisión garantizada.
Si gana la referencia constante, no se ha demostrado valor predictivo de los patrones.
No cambies parámetros mirando esta prueba sin reservar una nueva evaluación.

In [ ]:
test_predictions = test[['county_fips','target_year','base_year','y']].copy()
test_predictions['probability'] = selected_model.predict_proba(test[feature_cols])[:,1]
test_rows = []
for year, indices in test.groupby('target_year').groups.items():
    part = test.loc[indices]
    for name in [selected_name,'referencia_constante']:
        p = fitted[name].predict_proba(part[feature_cols])[:,1]
        test_rows.append({'modelo':name,'year':int(year),'n':len(part),
                          'prevalencia':float(part.y.mean()),**metrics(part.y,p)})
test_metrics = pd.DataFrame(test_rows).drop_duplicates(['modelo','year'])
display(test_metrics)
fig, axes = plt.subplots(1,2,figsize=(12,4))
CalibrationDisplay.from_predictions(test.y,test_predictions.probability,n_bins=10,
    strategy='quantile',ax=axes[0],name=selected_name)
axes[0].set_title('Calibración observada · prueba histórica')
axes[1].hist(test_predictions.probability,bins=20,color='#1976d2')
axes[1].set(title='Distribución de probabilidades · prueba',xlabel='Probabilidad',ylabel='Filas')
plt.tight_layout(); plt.show()
display(pd.DataFrame(confusion_matrix(test.y,test_predictions.probability >= CLASSIFICATION_THRESHOLD),
    index=['Real 0','Real 1'],columns=['Predicho 0','Predicho 1']))

## 5. Reentrenamiento y predicción del año objetivo
Se congela la elección y se reentrena con las etiquetas disponibles hasta 2025.
Se emite una predicción para cada código incluido con registro NOAA en el año base 2025.
Los condados sin ese registro no reciben automáticamente un porcentaje de cero: no se exportan.
Los porcentajes son resultados experimentales sin calibración adicional y con sesgo de cobertura.
Cambiar el año objetivo exige volver a ejecutar desde la configuración para mantener el desfase.

In [ ]:
final_model = clone(templates[selected_name]).fit(dataset[feature_cols],dataset.y)
eligible = storms.loc[storms.year == LAST_COMPLETE_YEAR,['county_fips']].drop_duplicates()
requests = eligible.assign(target_year=TARGET_YEAR)
future = make_features(requests)
assert future.storm_event_count_lag0.notna().all()
predictions = future[['county_fips','target_year','base_year','noaa_history_years']].copy()
predictions['probability'] = final_model.predict_proba(future[feature_cols])[:,1]
predictions['state_fips'] = future.state_fips
predictions['data_cutoff'] = f'{LAST_COMPLETE_YEAR}-12-31'
predictions['model_version'] = 'experimental-v1'
predictions['event_definition'] = 'NOAA flood_related_event_count > 0 (definicion amplia)'
predictions['is_demo'] = False  # Modelo entrenado, NO significa validación operativa.
predictions['is_experimental'] = True
assert predictions.probability.between(0,1).all()
assert not predictions.duplicated(['county_fips','target_year']).any()
display(predictions.sort_values('probability',ascending=False).head(15))
print('Predicciones:',len(predictions),'· Año:',TARGET_YEAR)

In [ ]:
# Consulta un condado por FIPS, conservando ceros iniciales.
COUNTY_FIPS = '48201'  # Harris, Texas. Cambia este código para consultar otro.
result = predictions.loc[predictions.county_fips == COUNTY_FIPS]
if result.empty:
    print('Sin predicción para este código dentro de la cobertura seleccionada.')
else:
    display(result)
    print(f'Probabilidad experimental: {result.iloc[0].probability:.1%}')
    print('Evento relacionado con inundaciones; no exclusivamente inundación.')

## 6. Exportación para la aplicación
El ZIP incluye CSV de predicciones, tabla preparada, métricas, modelo y metadatos.
La aplicación puede consultar el CSV o importarlo a SQLite; esta celda **no modifica la API ni
conecta automáticamente el dashboard**. Los códigos FIPS deben cargarse como texto.
Solo abre modelos `joblib` que tú hayas generado: ese formato puede ejecutar código al cargarse.
El archivo de modelo requiere las mismas versiones de dependencias registradas en los metadatos.

Los archivos de Colab son temporales: descarga el resultado antes de cerrar la sesión.

In [ ]:
OUTPUT_DIR = Path(globals().get('OUTPUT_DIR', 'floodrisk_modelo_v1'))
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
predictions.to_csv(OUTPUT_DIR/'predicciones.csv',index=False)
dataset.to_csv(OUTPUT_DIR/'tabla_entrenamiento.csv',index=False)
validation_metrics.to_csv(OUTPUT_DIR/'metricas_validacion.csv',index=False)
test_metrics.to_csv(OUTPUT_DIR/'metricas_prueba.csv',index=False)
test_predictions.to_csv(OUTPUT_DIR/'predicciones_prueba.csv',index=False)
joblib.dump(final_model,OUTPUT_DIR/'modelo.joblib')
metadata = {
    'generated_at_utc':datetime.now(timezone.utc).isoformat(),
    'model_name':selected_name,'model_version':'experimental-v1','target_year':TARGET_YEAR,
    'last_feature_year':LAST_COMPLETE_YEAR,'horizon_years':HORIZON,
    'label_training_end':int(dataset.target_year.max()),'feature_columns':feature_cols,
    'validation_year':VALIDATION_YEAR,'test_years':TEST_YEARS,
    'selection_metric':'validation_brier','threshold':CLASSIFICATION_THRESHOLD,
    'source_zip_sha256':zip_sha256,'source_generated_at':manifest.get('generated_at_utc'),
    'versions':{'pandas':pd.__version__,'numpy':np.__version__,'sklearn':sklearn.__version__,'joblib':joblib.__version__},
    'n_predictions':len(predictions),'random_seed':SEED,
    'limitations':[
        'Etiqueta amplia: incluye eventos relacionados, no solo inundaciones.',
        'Población condicionada a filas NOAA en año base y objetivo durante evaluación.',
        'No es probabilidad validada para todos los condados ni todos los eventos reales.',
        'Sin validación completa de catálogo FIPS ni instantáneas de publicación histórica.',
        'Probabilidades sin calibración adicional; 2026 parcial excluido.',
        'Modelo final reentrenado: métricas de prueba pertenecen a la versión previa al reentrenamiento.'
    ]
}
(OUTPUT_DIR/'metadatos.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
(OUTPUT_DIR/'LEEME.txt').write_text(
    'FloodRisk experimental. Leer metadatos.json antes de interpretar probabilidades.\n'
    'No es pronóstico meteorológico ni herramienta operativa.\n'
    'Cargar county_fips como texto. La integración con el dashboard sigue pendiente.\n',encoding='utf-8')
package = OUTPUT_DIR.parent/(OUTPUT_DIR.name+'.zip')
with zipfile.ZipFile(package,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTPUT_DIR.iterdir()):
        if p.is_file():
            z.write(p,arcname=p.name)
loaded_model = joblib.load(OUTPUT_DIR/'modelo.joblib')
assert np.allclose(loaded_model.predict_proba(future[feature_cols].head())[:,1], predictions.probability.head())
print('Modelo guardado y recargado correctamente:',package)

In [ ]:
# Descarga final en Colab. En local el ZIP queda en OUTPUT_DIR.parent.
try:
    from google.colab import files
except ImportError:
    print('Archivo local:',package.resolve())
else:
    files.download(str(package))

## Qué hacer después
1. Revisar métricas y si el modelo mejora la referencia; conservar también resultados desfavorables.
2. Decidir si esta etiqueta amplia basta para la demostración o conseguir eventos detallados para
   separar Flood/Flash Flood y otros tipos por condado y año.
3. Validar catálogo geográfico y cobertura antes de ampliar la población de predicción.
4. Importar predicciones a SQLite y cambiar la API para identificar versión, fecha, cobertura y límites.
5. Mostrar el porcentaje experimental sin presentar factores asociados como causas.

### Referencias técnicas
- [Google Colab: preguntas frecuentes](https://research.google.com/colaboratory/faq.html).
- [HistGradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html).
- [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).
- Fuentes de datos: ZIP proporcionado por el equipo, `data_dictionary.csv` y `source_manifest.json`.